# SASRec Training on Kaggle with 15 Negatives

This notebook trains SASRec model with:
- **15 negative samples** per position (vs paper's 1)
- **CUDA GPU** acceleration
- **RoPE (Rotary Position Embedding)** for better position encoding

## Setup Instructions
1. Upload your dataset file (e.g., `ml-1m.txt`) to Kaggle as a dataset
2. Add the dataset to this notebook
3. Enable GPU: Settings → Accelerator → GPU T4 x2
4. Run all cells

## 1. Install Dependencies & Setup

In [ ]:
!pip install -q polars scipy

import os
import sys
import time
import random
import math
from pathlib import Path
from typing import Any, Tuple, Optional, Callable, Dict, List
from collections import defaultdict

import numpy as np
import polars as pl
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Model Components (RoPE, FFN, Encoder)

In [ ]:
# ============= RoPE (Rotary Position Embedding) =============
class RotaryEmbedding(torch.nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 1024) -> None:
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        
        inv_freq: torch.Tensor = 1.0 / (10000 ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.cached_cos: torch.Tensor | None = None
        self.cached_sin: torch.Tensor | None = None
        
    def forward(self, seq_len: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.cached_cos is None or seq_len > self.cached_cos.shape[1]:
            t: torch.Tensor = torch.arange(seq_len, dtype=torch.float32, device=device)
            freqs: torch.Tensor = torch.einsum("i,j->ij", t, self.inv_freq)
            emb: torch.Tensor = torch.cat((freqs, freqs), dim=-1)
            self.cached_cos = emb.cos()[None, :, None, :]
            self.cached_sin = emb.sin()[None, :, None, :]
        
        assert self.cached_cos is not None and self.cached_sin is not None
        return self.cached_cos[:, :seq_len, ...], self.cached_sin[:, :seq_len, ...]

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

# ============= Feed-Forward Network =============
class PointWiseFeedForward(torch.nn.Module):
    def __init__(self, hidden_units: int, dropout_rate: float) -> None:
        super(PointWiseFeedForward, self).__init__()
        self.conv1: torch.nn.Conv1d = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1) 
        self.dropout1: torch.nn.Dropout = torch.nn.Dropout(p=dropout_rate)
        self.act: torch.nn.GELU = torch.nn.GELU() 
        self.conv2: torch.nn.Conv1d = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2: torch.nn.Dropout = torch.nn.Dropout(p=dropout_rate)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        x: torch.Tensor = inputs.transpose(-1, -2)
        x = self.conv1(x)
        x = self.dropout1(x)
        x = self.act(x)
        x = self.conv2(x)
        x = self.dropout2(x)
        outputs: torch.Tensor = x.transpose(-1, -2)
        return outputs

# ============= Encoder Layer with RoPE =============
class EncoderLayer(torch.nn.Module):
    def __init__(self, hidden_units: int, num_heads: int, dropout_rate: float) -> None:
        super().__init__()
        self.hidden_units: int = hidden_units
        self.num_heads: int = num_heads
        self.head_dim: int = hidden_units // num_heads

        assert self.head_dim * num_heads == hidden_units, "Hidden units must be divisible by num_heads"
        
        self.W_q: torch.nn.Linear = torch.nn.Linear(hidden_units, hidden_units, bias=False)
        self.W_k: torch.nn.Linear = torch.nn.Linear(hidden_units, hidden_units, bias=False)
        self.W_v: torch.nn.Linear = torch.nn.Linear(hidden_units, hidden_units, bias=False)
        self.out_proj: torch.nn.Linear = torch.nn.Linear(hidden_units, hidden_units)
        self.dropout: torch.nn.Dropout = torch.nn.Dropout(dropout_rate)
        
    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor], rotary_emb_fn: Optional[Callable] = None) -> torch.Tensor:
        B, L, H = x.shape
        
        q = self.W_q(x).view(B, L, self.num_heads, self.head_dim)
        k = self.W_k(x).view(B, L, self.num_heads, self.head_dim)
        v = self.W_v(x).view(B, L, self.num_heads, self.head_dim)

        if rotary_emb_fn is not None:
            cos, sin = rotary_emb_fn(L, x.device)
            q, k = apply_rotary_pos_emb(q, k, cos, sin)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, torch.finfo(scores.dtype).min)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).contiguous().view(B, L, H)
        output = self.out_proj(context)
        return output

print("✓ Model components loaded")

## 3. SASRec Model

In [ ]:
class SASRec(torch.nn.Module):
    def __init__(self, user_num: int, item_num: int, args: Any) -> None:
        super(SASRec, self).__init__()

        self.user_num: int = user_num
        self.item_num: int = item_num
        self.dev: torch.device = args.device
        self.norm_first: bool = args.norm_first

        self.item_emb: torch.nn.Embedding = torch.nn.Embedding(self.item_num+1, args.hidden_units, padding_idx=0)
        
        head_dim: int = args.hidden_units // args.num_heads
        self.rope: RotaryEmbedding = RotaryEmbedding(head_dim, max_seq_len=args.maxlen)

        self.emb_dropout: torch.nn.Dropout = torch.nn.Dropout(p=args.dropout_rate)

        self.attention_layernorms: torch.nn.ModuleList = torch.nn.ModuleList()
        self.attention_layers: torch.nn.ModuleList = torch.nn.ModuleList()
        self.forward_layernorms: torch.nn.ModuleList = torch.nn.ModuleList()
        self.forward_layers: torch.nn.ModuleList = torch.nn.ModuleList()

        self.last_layernorm: torch.nn.LayerNorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)

        for _ in range(args.num_blocks):
            new_attn_layernorm: torch.nn.LayerNorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.attention_layernorms.append(new_attn_layernorm)

            new_attn_layer: EncoderLayer = EncoderLayer(
                args.hidden_units,
                args.num_heads,
                args.dropout_rate
            )
            self.attention_layers.append(new_attn_layer)

            new_fwd_layernorm: torch.nn.LayerNorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.forward_layernorms.append(new_fwd_layernorm)

            new_fwd_layer: PointWiseFeedForward = PointWiseFeedForward(args.hidden_units, args.dropout_rate)
            self.forward_layers.append(new_fwd_layer)
            
    def log2feats(self, log_seqs: Any) -> torch.Tensor:
        seqs: torch.Tensor = self.item_emb(torch.as_tensor(log_seqs, dtype=torch.long, device=self.dev))
        seqs *= self.item_emb.embedding_dim ** 0.5
        seqs = self.emb_dropout(seqs)

        tl: int = seqs.shape[1]
        attention_mask: torch.Tensor = torch.tril(torch.ones((tl, tl), device=self.dev))

        for i in range(len(self.attention_layers)):
            if self.norm_first:
                x: torch.Tensor = self.attention_layernorms[i](seqs)
                mha_outputs: torch.Tensor = self.attention_layers[i](
                    x, 
                    attn_mask=attention_mask, 
                    rotary_emb_fn=self.rope
                )
                seqs = seqs + mha_outputs
                seqs = seqs + self.forward_layers[i](self.forward_layernorms[i](seqs))
            else:
                mha_outputs: torch.Tensor = self.attention_layers[i](
                    seqs, 
                    attn_mask=attention_mask, 
                    rotary_emb_fn=self.rope
                )
                seqs = self.attention_layernorms[i](seqs + mha_outputs)
                seqs = self.forward_layernorms[i](seqs + self.forward_layers[i](seqs))

        log_feats: torch.Tensor = self.last_layernorm(seqs) 
        return log_feats
    
    def forward(self, user_ids: Any, log_seqs: Any, pos_seqs: Any, neg_seqs: Any) -> Tuple[torch.Tensor, torch.Tensor]:      
        log_feats: torch.Tensor = self.log2feats(log_seqs) 

        pos_embs: torch.Tensor = self.item_emb(torch.as_tensor(pos_seqs, dtype=torch.long, device=self.dev))
        neg_ids: torch.Tensor = torch.as_tensor(neg_seqs, dtype=torch.long, device=self.dev)
        neg_embs: torch.Tensor = self.item_emb(neg_ids)

        pos_logits: torch.Tensor = (log_feats * pos_embs).sum(dim=-1)

        if neg_embs.dim() == 3:
            neg_logits: torch.Tensor = (log_feats * neg_embs).sum(dim=-1)
        else:
            neg_logits = (log_feats.unsqueeze(-2) * neg_embs).sum(dim=-1)

        return pos_logits, neg_logits
    
    def predict(self, user_ids: Any, log_seqs: Any, item_indices: Any) -> torch.Tensor:
        log_feats: torch.Tensor = self.log2feats(log_seqs) 
        final_feat: torch.Tensor = log_feats[:, -1, :] 
        item_embs: torch.Tensor = self.item_emb(torch.as_tensor(item_indices, dtype=torch.long, device=self.dev)) 
        logits: torch.Tensor = item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1)
        return logits

print("✓ SASRec model loaded")

## 4. Data Preprocessing & Dataset

In [ ]:
def convert_to_bin(dataset_name: str, data_dir: str = '/kaggle/input'):
    """Convert text data to binary format for faster loading"""
    data_path = f'{data_dir}/{dataset_name}/{dataset_name}.txt'
    bin_dir = f'bins/{dataset_name}_bin'
    os.makedirs(bin_dir, exist_ok=True)
    
    q = (
        pl.scan_csv(
            data_path,
            separator=' ',
            has_header=False,
            new_columns=['uid', 'iid'],
            schema={'uid': pl.Int32, 'iid': pl.Int32}
        ).sort('uid')
    )
    
    df = q.collect()
    all_items = df['iid'].to_numpy()
    user_stats = df.group_by('uid', maintain_order=True).len()
    
    unique_uids = user_stats['uid'].to_numpy()
    lengths = user_stats['len'].to_numpy()
    
    offsets = np.zeros(len(lengths), dtype=np.int64)
    offsets[1:] = np.cumsum(lengths)[:-1]
    
    max_user = unique_uids.max()
    max_item = all_items.max()
    
    user_index = np.zeros((max_user + 1, 2), dtype=np.int64)
    for i, uid in enumerate(unique_uids):
        user_index[uid, 0] = offsets[i]
        user_index[uid, 1] = lengths[i]
    
    np.save(f'{bin_dir}/all_items.npy', all_items)
    np.save(f'{bin_dir}/user_index.npy', user_index)
    
    with open(f'{bin_dir}/meta.txt', 'w') as f:
        f.write(f'{max_user},{max_item}')
    
    print(f"Converted: {len(unique_uids)} users, {max_item} items")

def load_metadata(dataset_name: str) -> Tuple[int, int]:
    bin_dir = Path(f'bins/{dataset_name}_bin')
    meta_path = bin_dir / 'meta.txt'
    with open(meta_path, 'r') as f:
        usernum, itemnum = map(int, f.read().strip().split(','))
    return usernum, itemnum

def data_partition(fname: str, data_dir: str = '/kaggle/input') -> Tuple[Dict, Dict, Dict, int, int]:
    usernum = 0
    itemnum = 0
    User = defaultdict(list)
    user_train = {}
    user_valid = {}
    user_test = {}
    
    data_path = f'{data_dir}/{fname}/{fname}.txt'
    with open(data_path, 'r') as f:
        for line in f:
            u, i = line.rstrip().split(' ')
            u = int(u)
            i = int(i)
            usernum = max(u, usernum)
            itemnum = max(i, itemnum)
            User[u].append(i)
    
    for user in User:
        nfeedback = len(User[user])
        if nfeedback < 4:
            user_train[user] = User[user]
            user_valid[user] = []
            user_test[user] = []
        else:
            user_train[user] = User[user][:-2]
            user_valid[user] = [User[user][-2]]
            user_test[user] = [User[user][-1]]
    
    return user_train, user_valid, user_test, usernum, itemnum

class SASRecDataset(Dataset):
    def __init__(self, dataset_name: str, maxlen: int, mode: str = 'train', num_negatives: int = 1):
        self.dataset_name = dataset_name
        self.maxlen = maxlen
        self.mode = mode
        self.num_negatives = max(1, int(num_negatives))
        
        bin_dir = Path(f'bins/{dataset_name}_bin')
        self.all_items = np.load(bin_dir / 'all_items.npy', mmap_mode='r')
        self.user_index = np.load(bin_dir / 'user_index.npy', mmap_mode='r')
        self.usernum, self.itemnum = load_metadata(dataset_name)
        
        self.valid_users = np.where(self.user_index[:, 1] > 0)[0]
        if mode == 'train':
            self.valid_users = self.valid_users[self.user_index[self.valid_users, 1] >= 4]
            np.random.shuffle(self.valid_users)
    
    def __len__(self) -> int:
        return len(self.valid_users)
    
    def _get_user_sequence(self, uid: int) -> np.ndarray:
        offset, length = self.user_index[uid]
        return np.array(self.all_items[offset:offset + length], dtype=np.int32)
    
    def _random_neq(self, l: int, r: int, s: set) -> int:
        t = np.random.randint(l, r)
        while t in s:
            t = np.random.randint(l, r)
        return t
    
    def __getitem__(self, idx: int) -> Tuple[int, np.ndarray, np.ndarray, np.ndarray]:
        uid = self.valid_users[idx]
        full_seq = self._get_user_sequence(uid)
        
        if self.mode == 'train':
            seq_items = full_seq[:-2] if len(full_seq) >= 4 else full_seq
        elif self.mode == 'valid':
            seq_items = full_seq[:-1]
        else:
            seq_items = full_seq
        
        if len(seq_items) <= 1:
             return (
                 uid,
                 np.zeros(self.maxlen, dtype=np.int32),
                 np.zeros(self.maxlen, dtype=np.int32),
                 np.zeros((self.maxlen, self.num_negatives), dtype=np.int32),
             )
        
        seq = np.zeros(self.maxlen, dtype=np.int32)
        pos = np.zeros(self.maxlen, dtype=np.int32)
        neg = np.zeros((self.maxlen, self.num_negatives), dtype=np.int32)
        
        nxt = seq_items[-1]
        idx = self.maxlen - 1
        ts = set(seq_items)
        
        for i in reversed(seq_items[:-1]):
            seq[idx] = i
            pos[idx] = nxt
            for j in range(self.num_negatives):
                neg[idx, j] = self._random_neq(1, self.itemnum + 1, ts)
            nxt = i
            idx -= 1
            if idx == -1: break
        
        return uid, seq, pos, neg

def get_dataloader(dataset_name, maxlen, batch_size, mode='train', num_workers=2, num_negatives: int = 1):
    dataset = SASRecDataset(dataset_name, maxlen, mode, num_negatives=num_negatives)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=(mode == 'train'),
        num_workers=num_workers, pin_memory=True, drop_last=False
    )

print("✓ Data utilities loaded")

## 5. Evaluation Functions

In [ ]:
def _batch_evaluate_logic(model, dataset, args, mode='test'):
    [train, valid, test, usernum, itemnum] = dataset
    
    NDCG = 0.0
    HT = 0.0
    valid_user = 0.0
    
    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = list(range(1, usernum + 1))

    eval_batch_size = 100 
    
    for start_idx in range(0, len(users), eval_batch_size):
        end_idx = min(start_idx + eval_batch_size, len(users))
        batch_users = users[start_idx:end_idx]
        
        batch_u_ids = []
        batch_seqs = []
        batch_item_indices = []
        
        for i, u in enumerate(batch_users):
            if len(train[u]) < 1 or (mode == 'test' and len(test[u]) < 1) or (mode == 'valid' and len(valid[u]) < 1):
                continue
            
            seq = np.zeros([args.maxlen], dtype=np.int32)
            idx = args.maxlen - 1
            
            if mode == 'test':
                seq[idx] = valid[u][0]
                idx -= 1
                source_seq = train[u]
            else:
                source_seq = train[u]
                
            for item in reversed(source_seq):
                seq[idx] = item
                idx -= 1
                if idx == -1: break
            
            rated = set(train[u])
            rated.add(0)
            
            if mode == 'test':
                target_item = test[u][0]
            else:
                target_item = valid[u][0]
                
            item_idx = [target_item]
            
            for _ in range(100):
                t = np.random.randint(1, itemnum + 1)
                while t in rated:
                    t = np.random.randint(1, itemnum + 1)
                item_idx.append(t)
                
            batch_u_ids.append(u)
            batch_seqs.append(seq)
            batch_item_indices.append(item_idx)

        if len(batch_u_ids) == 0:
            continue

        np_u_ids = np.array(batch_u_ids)
        np_seqs = np.array(batch_seqs)
        np_items = np.array(batch_item_indices)
        
        predictions = model.predict(np_u_ids, np_seqs, np_items)
        target_scores = predictions[:, 0]
        ranks = (predictions > target_scores.unsqueeze(1)).sum(dim=1)
        ranks = ranks.cpu().numpy()
        
        valid_user += len(ranks)
        hits = (ranks < 10).astype(np.float32)
        ndcgs = hits * (1.0 / np.log2(ranks + 2.0))
        
        HT += hits.sum()
        NDCG += ndcgs.sum()
        
        if (start_idx // eval_batch_size) % 10 == 0:
            print('.', end="")
            sys.stdout.flush()

    return NDCG / valid_user, HT / valid_user

def evaluate(model, dataset: Tuple, args) -> Tuple[float, float]:
    return _batch_evaluate_logic(model, dataset, args, mode='test')

def evaluate_valid(model, dataset: Tuple, args) -> Tuple[float, float]:
    return _batch_evaluate_logic(model, dataset, args, mode='valid')

print("✓ Evaluation functions loaded")

## 6. Configuration & Hyperparameters

In [ ]:
class Args:
    def __init__(self):
        # Dataset
        self.dataset = 'ml-1m'  # Change this to your dataset name
        self.data_dir = '/kaggle/input'  # Kaggle input directory
        
        # Model architecture
        self.hidden_units = 50
        self.num_blocks = 2
        self.num_heads = 1
        self.maxlen = 200
        self.dropout_rate = 0.2
        self.norm_first = False
        
        # Training
        self.batch_size = 128
        self.lr = 0.001
        self.num_epochs = 200  # Adjust as needed
        self.l2_emb = 0.0
        self.num_negatives = 15  # ★ 15 negatives (vs paper's 1)
        
        # Device
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.use_amp = True if self.device == 'cuda' else False
        
        # DataLoader
        self.num_workers = 2
        
        # Output
        self.train_dir = 'kaggle_run'

args = Args()
print(f"Configuration:")
print(f"  Dataset: {args.dataset}")
print(f"  Device: {args.device}")
print(f"  Num Negatives: {args.num_negatives}")
print(f"  Hidden Units: {args.hidden_units}")
print(f"  Num Blocks: {args.num_blocks}")
print(f"  Batch Size: {args.batch_size}")
print(f"  Learning Rate: {args.lr}")
print(f"  Max Epochs: {args.num_epochs}")

## 7. Prepare Data

In [ ]:
# Convert dataset to binary format
bin_dir = Path(f'bins/{args.dataset}_bin')
if not bin_dir.exists() or not (bin_dir / 'all_items.npy').exists():
    print(f"Converting {args.dataset} to binary format...")
    convert_to_bin(args.dataset, data_dir=args.data_dir)
else:
    print(f"Binary data already exists for {args.dataset}")

# Load metadata
usernum, itemnum = load_metadata(args.dataset)
print(f"Users: {usernum}, Items: {itemnum}")

# Create dataloader
train_loader = get_dataloader(
    args.dataset, 
    args.maxlen, 
    args.batch_size, 
    mode='train',
    num_workers=args.num_workers,
    num_negatives=args.num_negatives,
)
print(f"Training batches per epoch: {len(train_loader)}")

# Partition data for evaluation
dataset = data_partition(args.dataset, data_dir=args.data_dir)
[user_train, user_valid, user_test, _, _] = dataset

cc = sum(len(user_train[u]) for u in user_train)
print(f'Average sequence length: {cc / len(user_train):.2f}')

## 8. Initialize Model & Optimizer

In [ ]:
# Create output directory
output_dir = f'{args.dataset}_{args.train_dir}'
os.makedirs(output_dir, exist_ok=True)

# Initialize model
model = SASRec(usernum, itemnum, args).to(args.device)

# Xavier initialization
for name, param in model.named_parameters():
    try:
        torch.nn.init.xavier_normal_(param.data)
    except:
        pass

model.item_emb.weight.data[0, :] = 0

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=args.lr,
    betas=(0.9, 0.98),
    weight_decay=0.01
)

# AMP scaler
use_amp = args.use_amp and args.device == 'cuda'
scaler = torch.amp.grad_scaler.GradScaler(device='cuda') if use_amp else None

print(f"Model initialized on {args.device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"AMP enabled: {use_amp}")

## 9. Training Loop

In [ ]:
# Training setup
f = open(os.path.join(output_dir, 'log.txt'), 'w')
f.write('epoch (val_ndcg, val_hr) (test_ndcg, test_hr)\n')

best_val_ndcg = 0.0
best_test_ndcg = 0.0
T = 0.0
t0 = time.time()

print("\n" + "="*70)
if use_amp:
    print("Starting Training with AMP (Automatic Mixed Precision)")
else:
    print("Starting Training (FP32)")
print("="*70)
print(f"Model: SASRec-RoPE | Dataset: {args.dataset}")
print(f"Hidden: {args.hidden_units} | Layers: {args.num_blocks} | Heads: {args.num_heads}")
print(f"Max Length: {args.maxlen} | Batch Size: {args.batch_size} | LR: {args.lr}")
print(f"Num Negatives: {args.num_negatives}")
print("="*70 + "\n")

for epoch in range(1, args.num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{args.num_epochs}", unit="batch")
    for step, batch in enumerate(pbar):
        u, seq, pos, neg = batch
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast_mode.autocast(device_type='cuda'):
                pos_logits, neg_logits = model(u, seq, pos, neg)
                indices = np.where(pos != 0)

                pos_selected = pos_logits[indices]
                neg_selected = neg_logits[indices]
                if neg_selected.dim() == 1:
                    neg_selected = neg_selected.unsqueeze(1)
                cand_logits = torch.cat([pos_selected.unsqueeze(1), neg_selected], dim=1)
                loss = (-F.log_softmax(cand_logits, dim=1)[:, 0]).mean()

                for param in model.item_emb.parameters():
                    loss += args.l2_emb * torch.sum(param ** 2)

            scaler.scale(loss).backward()
            scaler.step(optimizer=optimizer)
            scaler.update()
        else:
            pos_logits, neg_logits = model(u, seq, pos, neg)
            indices = np.where(pos != 0)

            pos_selected = pos_logits[indices]
            neg_selected = neg_logits[indices]
            if neg_selected.dim() == 1:
                neg_selected = neg_selected.unsqueeze(1)
            cand_logits = torch.cat([pos_selected.unsqueeze(1), neg_selected], dim=1)
            loss = (-F.log_softmax(cand_logits, dim=1)[:, 0]).mean()

            for param in model.item_emb.parameters():
                loss += args.l2_emb * torch.sum(param ** 2)

            loss.backward()
            optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg_loss': f'{epoch_loss/num_batches:.4f}'})
    
    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch:3d} | Avg Loss: {avg_loss:.4f}', end='')
    
    # Evaluate every 20 epochs
    if epoch % 20 == 0:
        model.eval()
        t1 = time.time() - t0
        T += t1
        
        with torch.no_grad():
            with torch.amp.autocast_mode.autocast(device_type='cuda', enabled=False):
                t_test = evaluate(model, dataset, args)
                t_valid = evaluate_valid(model, dataset, args)
        
        print(f' | Time: {T:.1f}s')
        print(f'         Valid - NDCG@10: {t_valid[0]:.4f}, HR@10: {t_valid[1]:.4f}')
        print(f'         Test  - NDCG@10: {t_test[0]:.4f}, HR@10: {t_test[1]:.4f}')
        
        is_best = False
        if t_valid[0] > best_val_ndcg or t_test[0] > best_test_ndcg:
            best_val_ndcg = max(t_valid[0], best_val_ndcg)
            best_test_ndcg = max(t_test[0], best_test_ndcg)
            is_best = True
            
            fname = f'SASRec.epoch={epoch}.lr={args.lr}.layer={args.num_blocks}.head={args.num_heads}.hidden={args.hidden_units}.maxlen={args.maxlen}.pth'
            torch.save(model.state_dict(), os.path.join(output_dir, fname))
            print(f'         ✓ New best model saved: {fname}')
        
        if is_best:
            print(f'         Best so far - Valid NDCG: {best_val_ndcg:.4f}, Test NDCG: {best_test_ndcg:.4f}')
        
        f.write(f'{epoch} {t_valid} {t_test}\n')
        f.flush()
        t0 = time.time()
        print()
    else:
        print()

# Save final model
fname = f'SASRec.epoch={args.num_epochs}.lr={args.lr}.layer={args.num_blocks}.head={args.num_heads}.hidden={args.hidden_units}.maxlen={args.maxlen}.pth'
torch.save(model.state_dict(), os.path.join(output_dir, fname))
print(f"\nFinal model saved: {fname}")

f.close()
print("Training completed!")
print(f"\nBest Results:")
print(f"  Valid NDCG@10: {best_val_ndcg:.4f}")
print(f"  Test NDCG@10: {best_test_ndcg:.4f}")

## 10. Download Results

After training completes, you can download:
- Model checkpoints from the output directory
- Training log (`log.txt`)

The models will be in the `ml-1m_kaggle_run/` folder (or whatever your dataset name is).